# Setup


In [1]:
! pip install pillow
! pip install chromadb
! pip install opencv-python onnxruntime
! pip install insightface
! pip install FlagEmbedding

# ! pip install tensorflow
# ! pip install deepface
# ! pip install faiss-cpu
# ! pip install faiss-gpu
# ! pip install stanza
# ! pip install sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.9/131.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.7/65.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.9 MB/s eta 

In [ ]:
import json
import os
import pickle
import shutil
import time
# from copy import deepcopy
# from difflib import get_close_matches
# from io import BytesIO
from pathlib import Path
from typing import Any, Dict, List, Tuple, Union
from urllib.parse import urljoin

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import cv2
# import torchvision
# import torchvision.transforms as transforms
from insightface.app import FaceAnalysis
from PIL import Image

# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import DataLoader, Dataset, random_split

from FlagEmbedding import BGEM3FlagModel
# from huggingface_hub import login
# from peft import LoraConfig, get_peft_model
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForTokenClassification,
    # AutoProcessor,
    AutoTokenizer,
    # CLIPFeatureExtractor,
    # CLIPVisionModel,
    # Gemma3ForConditionalGeneration,
    # RobertaModel,
    pipeline,
)

import chromadb
from chromadb.config import Settings

import requests
# from bs4 import BeautifulSoup
from tqdm import tqdm

# Configs

In [3]:
class Config:
    TEXT_EMB_MODEL, TEXT_EMB_PROV = "BAAI/bge-m3", "bge"
    # TEXT_EMB_MODEL, TEXT_EMB_PROV = "jinaai/jina-embeddings-v2-base-en", "jina"
    NER_MODEL = "HooshvareLab/roberta-fa-zwnj-base-ner"

    PROJECT_PATH         = Path("/content/drive/MyDrive/SUT/NLP/Project")
    GT_IMG_DATASET_PATH  = PROJECT_PATH / "face_recognition_ground_truth_images"
    VAL_IMG_DATASET_PATH = PROJECT_PATH / "face_recognition_validation_images"

    CACHE_PATH = PROJECT_PATH / "cached_img_text_embeddings"

    IMG_EMBS_CACHE_PATH  = CACHE_PATH / "img_embeddings.pkl"
    TXT_EMBS_CACHE_PATH  = CACHE_PATH / f"txt_embeddings_{TEXT_EMB_PROV}.pkl"

    GT_IMG_JSON_PATH_1   = GT_IMG_DATASET_PATH / "0000_all_img_link_added_1.json"
    GT_IMG_JSON_PATH_3   = GT_IMG_DATASET_PATH / "0000_all_img_link_added_3.json"
    GT_IMG_JSON_PATH_4   = GT_IMG_DATASET_PATH / "0000_all_img_link_added_4.json"
    GT_IMG_JSON_PATH_7   = GT_IMG_DATASET_PATH / "0000_all_img_link_added_7.json"

# Utilites

In [4]:
def show_image_with_title(image_path: str, title_text: str = ""):
    try:
        img = Image.open(image_path)
        plt.imshow(img)
        plt.axis('off')
        plt.title(title_text)
        plt.show()
    except FileNotFoundError:
        print(f"Error: Could not display file at '{image_path}'")
    except Exception as e:
        print(f"An error occurred while displaying image: {e}")

In [5]:
def read_json_file(file_path: str):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"Successfully loaded data from {file_path}")
        return data
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None
    except json.JSONDecodeError:
        print(f"Error: The file at {file_path} is not a valid JSON file.")
        return None

In [6]:
def create_people_list(json_data: dict, json_id):
    people_list = []
    for idx, item in enumerate(json_data):
        item_summary = {"iteration": item["iteration"], "json_id": json_id}
        try:
            if type(item["json_pure"]["name"]) is list:
                person_name = " ".join(item["json_pure"]["name"]).replace('\u200c', ' ')
            else:
                person_name = item["json_pure"]["name"].replace('\u200c', ' ')
            item_summary["person_name"] = person_name
        except:
            continue
            item_summary["person_name"] = "unknown"
        people_list.append(item_summary)
    return people_list

In [7]:
def dict_pretty_print(data_to_print: list | dict):
    formatted_json = json.dumps(
        data_to_print,
        indent=4,
        ensure_ascii=False
    )
    print(formatted_json)

In [8]:
def get_info_from_dataset(json_id: str, iteration: int) -> dict:
    if json_id == "json_1":
        data_source = json_1
    elif json_id == "json_3":
        data_source = json_3
    elif json_id == "json_4":
        data_source = json_4
    elif json_id == "json_7":
        data_source = json_7

    if data_source[iteration]["iteration"] == iteration:
        result = data_source[iteration]
    else:
        for item in data_source:
            if item["iteration"] == iteration:
                result = item
                break
    return result

In [9]:
def extract_image_fields(data: Dict[str, Any]) -> Dict[str, List[Dict[str, str]]]:
    extracted_data = [
        {
            "url": image.get("url"),
            "description": image.get("description"),
            "idx": image.get("idx"),
        }
        for image in data.get("image_urls", [])
    ]
    return extracted_data

# Text Retrieval Pipeline

In [10]:
class PersonNameExtractor:
    def __init__(self, model_name: str = Config.NER_MODEL):
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.nlp_pipeline = pipeline(
            "ner",
            model=model,
            tokenizer=tokenizer,
            aggregation_strategy="simple"
        )

    def extract(self, text: str) -> List[str]:
        ner_results = self.nlp_pipeline(text)
        person_names = [
            entity['word']
            for entity in ner_results
            if entity['entity_group'] == 'PER'
        ]
        return person_names

In [11]:
class TextEmbedder:
    def __init__(self, model_name: str, model_provider: str):
        self.model_name = model_name
        self.provider = model_provider.lower()
        print(f"Initializing embedder with provider: '{self.provider}' and model: '{self.model_name}'...")
        if self.provider == 'jina':
            self.model = SentenceTransformer(self.model_name)
        elif self.provider == 'bge':
            self.model = BGEM3FlagModel(self.model_name, use_fp16=True)
        else:
            raise ValueError("Unsupported model_provider. Please choose 'jina' or 'bge'.")
        print("Model loaded successfully.")


    def _encode(self, texts: Union[str, List[str]]) -> np.ndarray:
        if self.provider == 'jina':
            return self.model.encode(texts, show_progress_bar=True)
        elif self.provider == 'bge':
            return self.model.encode(texts)['dense_vecs']


    def get_embedding(self, text: str) -> np.ndarray:
        return self._encode(text)


    def create_database_embeddings(self, source_data: List[Dict[str, Any]], start_index: int = 0) -> Tuple[List[str], List[dict], List[list]]:
        if not source_data:
            return [], [], []

        person_names = [item['person_name'] for item in source_data]
        ids = [f"text_id_{i + start_index}" for i in range(len(source_data))]

        metadatas = [
            {
                "person_name": item['person_name'],
                "iteration": item['iteration'],
                "json_id": item['json_id']
            } for item in source_data
        ]

        print(f"Generating embeddings for {len(person_names)} names with {self.provider} model...")
        embeddings = self._encode(person_names)

        return ids, metadatas, embeddings.tolist()

In [12]:
class TextVectorDatabase:
    def __init__(self, collection_name: str):
        self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )


    def add_batch(self, ids: List[str], embeddings: List[list], metadatas: List[dict], batch_size: int = 1000):
        if not embeddings or not ids:
            print("No embeddings or IDs provided.")
            return

        num_items = len(ids)
        for i in tqdm(range(0, num_items, batch_size), desc="Adding to Vector DB"):
            end_index = min(i + batch_size, num_items)

            self.collection.add(
                ids=ids[i:end_index],
                embeddings=embeddings[i:end_index],
                metadatas=metadatas[i:end_index]
            )

        print(f"Successfully added {num_items} total items to the collection '{self.collection.name}'.")


    def search(self, query_embedding: np.ndarray, k: int = 1) -> dict:
        return self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=k
        )

In [13]:
def init_text_pipeline(source_data: List[Dict[str, Any]]) -> Tuple[TextEmbedder, TextVectorDatabase]:
    print("Initializing Text Embedder...")
    embedder = TextEmbedder(model_name=Config.TEXT_EMB_MODEL, model_provider=Config.TEXT_EMB_PROV)

    ids, metadatas, embeddings = [], [], []

    if Config.TXT_EMBS_CACHE_PATH.exists():
        print(f"Loading cached embeddings from {Config.TXT_EMBS_CACHE_PATH}...")
        with open(Config.TXT_EMBS_CACHE_PATH, 'rb') as f:
            cached_data = pickle.load(f)
            ids, metadatas, embeddings = cached_data["ids"], cached_data["metadatas"], cached_data["embeddings"]

        cached_json_ids = {meta['json_id'] for meta in metadatas}
        new_items_to_process = [item for item in source_data if item['json_id'] not in cached_json_ids]

        if new_items_to_process:
            print(f"Found {len(new_items_to_process)} new items to embed.")
            new_ids, new_metadatas, new_embeddings = embedder.create_database_embeddings(
                new_items_to_process, start_index=len(ids)
            )

            ids.extend(new_ids)
            metadatas.extend(new_metadatas)
            embeddings.extend(new_embeddings)

            with open(Config.TXT_EMBS_CACHE_PATH, 'wb') as f:
                pickle.dump({"ids": ids, "metadatas": metadatas, "embeddings": embeddings}, f)
            print(f"Cache updated with new embeddings at {Config.TXT_EMBS_CACHE_PATH}")
        else:
            print("Cache is already up to date.")

    else:
        print(f"Cache not found. Building embeddings from scratch for {len(source_data)} items...")
        ids, metadatas, embeddings = embedder.create_database_embeddings(source_data)

        with open(Config.TXT_EMBS_CACHE_PATH, 'wb') as f:
            pickle.dump({"ids": ids, "metadatas": metadatas, "embeddings": embeddings}, f)
        print(f"Embeddings cached to {Config.TXT_EMBS_CACHE_PATH}")

    if not embeddings:
        print("No embeddings were generated or loaded. Exiting.")
        exit()

    print("Initializing Vector Database for text...")
    vector_db = TextVectorDatabase(collection_name=f"{Config.TEXT_EMB_PROV}_person_names")
    vector_db.add_batch(ids, embeddings, metadatas)

    print(f"Text vector database ready with {vector_db.collection.count()} entries.")
    return embedder, vector_db

In [14]:
def search_person_by_text(
        text_query: str,
        n_results: int,
        person_name_extractor: PersonNameExtractor,
        embedder: TextEmbedder,
        vector_db: TextVectorDatabase,
    ) -> list | None:
    person_names = person_name_extractor.extract(text_query)
    if not person_names:
        print("No person name found!")
        return None
    query_embedding = embedder.get_embedding(person_names[0])
    results = vector_db.search(query_embedding, k=n_results)

    best_matches = results['metadatas'][0]
    distances = results['distances'][0]

    search_results = []
    for idx, (metadata, distance) in enumerate(zip(best_matches, distances)):
        person_info = get_info_from_dataset(metadata["json_id"],
                                            metadata["iteration"])
        item = {
            "nth_match": idx + 1,
            "json_id": metadata["json_id"],
            "iteration": metadata["iteration"],
            "person_name": person_info["json_pure"]["name"],
            "distance": distance,
            # "json": person_info["json_refined"],
            "textual_content": person_info["textual_content_response"],
            # "question_and_answer": person_info["question_and_answer_response"],
            "image_data": extract_image_fields(person_info["image_data"]),
        }
        search_results.append(item)
    return search_results

# Image Retrieval Pipeline

In [15]:
class FaceRecognizer:
    def __init__(self, model_pack_name: str = 'buffalo_l'):
        self.app = FaceAnalysis(name=model_pack_name, providers=['CPUExecutionProvider'])
        self.app.prepare(ctx_id=0, det_size=(640, 640))

    def get_embedding(self, image_path: Path) -> np.ndarray | None:
        try:
            img = cv2.imread(str(image_path))
            if img is None:
                return None
            faces = self.app.get(img)
            if faces:
                return faces[0].normed_embedding
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
        return None

    def create_database_embeddings(self, GT_IMG_DATASET_PATH: Path) -> Tuple[List[str], List[list]]:
        paths = []
        embeddings = []
        image_files = [p for p in GT_IMG_DATASET_PATH.rglob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]

        for img_path in tqdm(image_files, desc="Building embedding database"):
            embedding = self.get_embedding(img_path)
            if embedding is not None:
                paths.append(str(img_path))
                embeddings.append(embedding.tolist())
        return paths, embeddings

    def get_embeddings_for_paths(self, image_paths: List[Path]) -> Tuple[List[str], List[list]]:
        paths = []
        embeddings = []

        for img_path in tqdm(image_paths, desc="Computing embeddings for new images"):
            embedding = self.get_embedding(img_path)
            if embedding is not None:
                paths.append(str(img_path))
                embeddings.append(embedding.tolist())

        return paths, embeddings

In [16]:
class ImageVectorDatabase:
    def __init__(self, collection_name: str = "face_db"):
        self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )

    def add_batch(self, paths: List[str], embeddings: List[list], batch_size: int = 4000):
        if not embeddings or not paths:
            print("No embeddings or paths provided.")
            return

        num_items = len(paths)

        ids = [f"id_{i}" for i in range(num_items)]
        metadatas = [{"path": path} for path in paths]

        for i in range(0, num_items, batch_size):
            end_index = min(i + batch_size, num_items)

            print(f"Adding batch of {end_index - i} items to the vector database...")

            batch_embeddings = embeddings[i:end_index]
            batch_metadatas = metadatas[i:end_index]
            batch_ids = ids[i:end_index]

            self.collection.add(
                embeddings=batch_embeddings,
                metadatas=batch_metadatas,
                ids=batch_ids
            )

        print(f"Successfully added {num_items} total items to the collection.")

    def search(self, query_embedding: np.ndarray, k: int = 1):
        return self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=k
        )

In [17]:
def init_image_pipeline():
    print("Initializing Face Recognizer...")
    recognizer = FaceRecognizer()

    Config.CACHE_PATH.mkdir(exist_ok=True)

    paths = []
    embeddings = []

    if Config.IMG_EMBS_CACHE_PATH.exists():
        print(f"Loading cached embeddings from {Config.IMG_EMBS_CACHE_PATH}...")
        with open(Config.IMG_EMBS_CACHE_PATH, 'rb') as f:
            data = pickle.load(f)
            paths, embeddings = data["paths"], data["embeddings"]

        cached_paths_set = set(paths)
        all_image_files = [str(p) for p in Config.GT_IMG_DATASET_PATH.rglob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
        new_image_paths = [Path(p) for p in all_image_files if p not in cached_paths_set]

        if new_image_paths:
            print(f"Found {len(new_image_paths)} new images to process.")
            new_paths, new_embeddings = recognizer.get_embeddings_for_paths(new_image_paths)

            paths.extend(new_paths)
            embeddings.extend(new_embeddings)

            print(f"Updating cache file at {Config.IMG_EMBS_CACHE_PATH}...")
            with open(Config.IMG_EMBS_CACHE_PATH, 'wb') as f:
                pickle.dump({"paths": paths, "embeddings": embeddings}, f)
            print("Cache updated successfully.")
        else:
            print("Cache is up to date. No new images found.")

    else:
        print(f"Cache not found. Building embeddings from scratch for all images in {Config.GT_IMG_DATASET_PATH}...")
        paths, embeddings = recognizer.create_database_embeddings(Config.GT_IMG_DATASET_PATH)

        if paths:
            with open(Config.IMG_EMBS_CACHE_PATH, 'wb') as f:
                pickle.dump({"paths": paths, "embeddings": embeddings}, f)
            print(f"Embeddings cached to {Config.IMG_EMBS_CACHE_PATH}")

    if not embeddings:
        print("No embeddings found in the database. Exiting.")
        exit()

    print("Initializing Vector Database...")
    vector_db = ImageVectorDatabase(collection_name="insightface_faces")
    vector_db.add_batch(paths, embeddings)
    print(f"Vector database ready with {vector_db.collection.count()} entries.")

    return recognizer, vector_db

In [18]:
def search_person_by_image(
        query_image_path: Path,
        n_results: int,
        recognizer: FaceRecognizer,
        vector_db: ImageVectorDatabase,
    ) -> list | None:
    query_embedding = recognizer.get_embedding(query_image_path)
    if query_embedding is None:
        print(f"Could not get embedding for {query_image_path}. No face found or image is invalid.")
        return None

    results = vector_db.search(query_embedding, k=n_results)

    if not results or not results['ids'][0]:
        print("No matches found in the database for this person.")
        return None

    best_matches = results['metadatas'][0]
    distances = results['distances'][0]

    search_results = []
    for idx, (metadata, distance) in enumerate(zip(best_matches, distances)):
        json_id = "json_" + metadata["path"].split("/")[-3].split("_")[-1]
        iteration = int(metadata["path"].split("/")[-2])
        person_info = get_info_from_dataset(json_id, iteration)

        item = {
            "nth_match": idx + 1,
            "json_id": json_id,
            "iteration": iteration,
            "person_name": person_info["json_pure"]["name"],
            "distance": distance,
            # "json": person_info["json_refined"],
            "textual_content": person_info["textual_content_response"],
            # "question_and_answer": person_info["question_and_answer_response"],
            "image_data": extract_image_fields(person_info["image_data"]),
            "match_image_path": metadata["path"],
        }
        search_results.append(item)
    return search_results

# Merging Data Sources

In [21]:
json_1 = read_json_file(Config.GT_IMG_JSON_PATH_1)
json_3 = read_json_file(Config.GT_IMG_JSON_PATH_3)
json_4 = read_json_file(Config.GT_IMG_JSON_PATH_4)
json_7 = read_json_file(Config.GT_IMG_JSON_PATH_7)

json_1_people_list = create_people_list(json_1, "json_1")
json_3_people_list = create_people_list(json_3, "json_3")
json_4_people_list = create_people_list(json_4, "json_4")
json_7_people_list = create_people_list(json_7, "json_7")


people_list_total = [*json_1_people_list,
                     *json_3_people_list,
                     *json_4_people_list,
                     *json_7_people_list]

Successfully loaded data from /content/drive/MyDrive/SUT/NLP/Project/face_recognition_ground_truth_images/0000_all_img_link_added_1.json
Successfully loaded data from /content/drive/MyDrive/SUT/NLP/Project/face_recognition_ground_truth_images/0000_all_img_link_added_3.json
Successfully loaded data from /content/drive/MyDrive/SUT/NLP/Project/face_recognition_ground_truth_images/0000_all_img_link_added_4.json
Successfully loaded data from /content/drive/MyDrive/SUT/NLP/Project/face_recognition_ground_truth_images/0000_all_img_link_added_7.json


# Initializing Text & Image Retrieval Pipelines

In [22]:
text_embedder, text_vector_db = init_text_pipeline(people_list_total)

Initializing Text Embedder...
Initializing embedder with provider: 'bge' and model: 'BAAI/bge-m3'...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

... (24 outputs left out)

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

Model loaded successfully.
Loading cached embeddings from /content/drive/MyDrive/SUT/NLP/Project/cached_img_text_embeddings/txt_embeddings_bge.pkl...
Cache is already up to date.
Initializing Vector Database for text...


Adding to Vector DB: 100%|██████████| 4/4 [00:03<00:00,  1.06it/s]

Successfully added 3564 total items to the collection 'bge_person_names'.
Text vector database ready with 3564 entries.


In [ ]:
face_recognizer, image_vector_db = init_image_pipeline()

Initializing Face Recognizer...
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:04<00:00, 60568.25KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
Loading cach

Computing embeddings for new images:  69%|██████▉   | 124/180 [01:42<00:47,  1.17it/s]

In [ ]:
person_name_extractor = PersonNameExtractor()

# Testing Integrated Multimodal RAG

## Text as a Query

In [26]:
sample_text = """
هومن سیدی بازیگر و کارگردان تئاتر و سینما و تلویزیون ۴۵ ساله اهل رشت بوده
 و از سال ۱۳۷۳ تا به حال در عرصه کارگردانی و بازیگری فعالیت می‌کند
 سریال وحشی و قورباغه و فیلم‌های مغزهای کوچک زنگ زده، خشم و هیاهو
  ، آفریقا و سیزده از آثار هومن سیدی می‌باشند
"""

text_retrieval_result = search_person_by_text(
    sample_text,
    1,
    person_name_extractor,
    text_embedder,
    text_vector_db,
)
dict_pretty_print(text_retrieval_result)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[
    {
        "nth_match": 1,
        "json_id": "json_3",
        "iteration": 20,
        "person_name": "هومن سیدی",
        "distance": 1.1920928955078125e-07,
        "textual_content": "هومن سیدی در تاریخ ۸ آذر ۱۳۵۹ در رشت، استان گیلان متولد شد. او در دهه‌های هشتاد تا کنون در عرصه هنر فعالیت داشته و حرفه‌های بازیگری، کارگردانی، فیلمنامه‌نویسی و تدوین‌گری را بر عهده داشته است.\n\nسیدی در فیلم‌های متعددی از جمله «یک تکه نان» (۱۳۸۳)، «پابرهنه در بهشت» (۱۳۸۴)، «چهارشنبه سوری» (۱۳۸۴)، «آنکه دریا می‌رود» (۱۳۸۵)، «زخم شانه حوا» (۱۳۸۶)، «یک وجب از آسمان» (۱۳۸۷)، «آفریقا» (۱۳۸۹)، «آزادراه» (۱۳۸۹)، «سیزده» (۱۳۹۱)، «خط ویژه» (۱۳۹۲)، «شب بیرون» (۱۳۹۲)، «بوفالو» (۱۳۹۳)، «در مدت معلوم» (۱۳۹۳)، «من دیه گو مارادونا هستم» (۱۳۹۳)، «اعترافات ذهن خطرناک من» (۱۳۹۳)، «نیم‌رخ‌ها» (۱۳۹۳)، «پل خواب» (۱۳۹۳)، «مادری» (۱۳۹۴)، «اینجا کسی نمی‌میرد» (۱۳۹۴)، «خشم و هیاهو» (۱۳۹۴)، «کمدی انسانی» (۱۳۹۵)، «آذر» (۱۳۹۵)، «مصادره» (۱۳۹۶)، «مغزهای کوچک زنگ‌زده» (۱۳۹۶) و «طلا» (۱۳۹۷) و «جنگ جهانی سوم» (۱۴۰۱) به ایفای 

## Image as a Query

In [27]:
sample_image_path = Config.PROJECT_PATH / "homan_seyedi.jpg"
image_retrieval_result = search_person_by_image(
    sample_image_path,
    1,
    face_recognizer,
    image_vector_db,
)
dict_pretty_print(image_retrieval_result)

[
    {
        "nth_match": 1,
        "json_id": "json_3",
        "iteration": 20,
        "person_name": "هومن سیدی",
        "distance": 0.0,
        "textual_content": "هومن سیدی در تاریخ ۸ آذر ۱۳۵۹ در رشت، استان گیلان متولد شد. او در دهه‌های هشتاد تا کنون در عرصه هنر فعالیت داشته و حرفه‌های بازیگری، کارگردانی، فیلمنامه‌نویسی و تدوین‌گری را بر عهده داشته است.\n\nسیدی در فیلم‌های متعددی از جمله «یک تکه نان» (۱۳۸۳)، «پابرهنه در بهشت» (۱۳۸۴)، «چهارشنبه سوری» (۱۳۸۴)، «آنکه دریا می‌رود» (۱۳۸۵)، «زخم شانه حوا» (۱۳۸۶)، «یک وجب از آسمان» (۱۳۸۷)، «آفریقا» (۱۳۸۹)، «آزادراه» (۱۳۸۹)، «سیزده» (۱۳۹۱)، «خط ویژه» (۱۳۹۲)، «شب بیرون» (۱۳۹۲)، «بوفالو» (۱۳۹۳)، «در مدت معلوم» (۱۳۹۳)، «من دیه گو مارادونا هستم» (۱۳۹۳)، «اعترافات ذهن خطرناک من» (۱۳۹۳)، «نیم‌رخ‌ها» (۱۳۹۳)، «پل خواب» (۱۳۹۳)، «مادری» (۱۳۹۴)، «اینجا کسی نمی‌میرد» (۱۳۹۴)، «خشم و هیاهو» (۱۳۹۴)، «کمدی انسانی» (۱۳۹۵)، «آذر» (۱۳۹۵)، «مصادره» (۱۳۹۶)، «مغزهای کوچک زنگ‌زده» (۱۳۹۶) و «طلا» (۱۳۹۷) و «جنگ جهانی سوم» (۱۴۰۱) به ایفای نقش پرداخته یا آن‌ه

## Final Function

In [98]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get('Gemeni_API_KEY')

def _infer_mime_type_from_path(path: str) -> str:
    """
    Helper to guess mime_type from file extension.
    Extend this if needed (png, jpg, jpeg, gif, etc).
    """
    lower = path.lower()
    if lower.endswith(".jpg") or lower.endswith(".jpeg"):
        return "image/jpeg"
    elif lower.endswith(".png"):
        return "image/png"
    elif lower.endswith(".gif"):
        return "image/gif"
    else:
        # default fallback
        return "application/octet-stream"


SYSTEM_PROMPT_IRANIAN_FIGURES = """
شما یک مدل هوش مصنوعی هستید که تخصص‌تان در پاسخ‌دهی درباره مشاهیر ایرانی است.

اگر در زمینه اطلاعات مرتبط برای پاسخ به سوال وجود دارد از آن استفاده کن و اگر مرتبط نیست یا وجود ندارد از دانش عمومی خودت استفاده کن.

سوال 4 گزینه است لطفا فقط یکی از گزینه ها را انتخاب کن و فقط در یک کلمه یک عدد از 1 تا 4 انتخاب کن.
"""

def call_generative_model(
    input_text: str | None,
    input_image_path: str | None,
    retrieved_texts: list[str] | None = None,
    retrieved_images_paths: list[str] | None = None,
    model_name: str = "gemini-2.5-flash",
    temperature: float = 0.25,
    top_p: float = 0.5,
    max_output_tokens: int = 256,
    thinking_budget: int = 0
) -> str:
    """
    Calls a Gemini generative model with optional text & image context,
    incorporating retrieved texts/images, and optional main image + text.

    Either or both of input_text / input_image_path can be None,
    similarly retrieved_texts or retrieved_images_paths can be None.

    Args:
        model_name: name of the Gemini model, e.g. "gemini-2.5-flash"
        input_text: main user prompt / query (may be None if prompt only image)
        input_image_path: path to image file for main prompt (may be None)
        retrieved_texts: list of text strings retrieved by retriever (or None)
        retrieved_images_paths: list of image file paths retrieved as context (or None)
        temperature, top_p, max_output_tokens, thinking_budget: config params

    Returns:
        Generated text from the model.

    Raises:
        ValueError if both input_text and input_image_path are None (no prompt).
    """

    # Basic checks
    if input_text is None and input_image_path is None:
        raise ValueError("At least one of input_text or input_image_path must be provided for prompt.")

    client = genai.Client()

    contents: list[types.Part | str] = []

    # 1. Retrieved text context
    if retrieved_texts:
        # Format nicely, maybe separate them
        # text_context = "Context from retrieved texts:\n" + "\n---\n".join(retrieved_texts)
        text_context = "زمینه مرتبط از متون بازیابی‌شده:\n" + "\n---\n".join(retrieved_texts)
        contents.append(text_context)

    # 2. Retrieved images as context
    if retrieved_images_paths:
        for img_path in retrieved_images_paths:
            with open(img_path, "rb") as f:
                img_bytes = f.read()
            part = types.Part.from_bytes(
                data=img_bytes,
                mime_type=_infer_mime_type_from_path(img_path)
            )
            contents.append(part)

    # 3. Main input image
    if input_image_path:
        with open(input_image_path, "rb") as f:
            main_img_bytes = f.read()
        main_img_part = types.Part.from_bytes(
            data=main_img_bytes,
            mime_type=_infer_mime_type_from_path(input_image_path)
        )
        contents.append(main_img_part)

    # 4. Main input text
    if input_text:
        contents.append(input_text)

    # 5. Generation config
    config = types.GenerateContentConfig(
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=max_output_tokens,
        thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget),
        # system_instruction=SYSTEM_PROMPT_IRANIAN_FIGURES
    )

    # 6. Call Gemini
    response = client.models.generate_content(
        model=model_name,
        contents=contents,
        config=config
    )

    return response.text


In [41]:
# from call_generative_model  import call_generative_model

ans = call_generative_model(
        input_text="سلامممم",
        input_image_path=None,
        model_name="gemini-2.5-flash"
    )
ans

'سلام! چطوری؟ 😊'

In [100]:
def RAG(query, image_url=None):

    # retrieved_texts = search_by_text(query=query, k=3)["metadatas"][0]
    text_retrieve_result = search_person_by_text(
        query,
        1,
        person_name_extractor,
        text_embedder,
        text_vector_db
    )

    if text_retrieve_result:
        retrieved_texts = text_retrieve_result[0]["textual_content"]
    else:
        retrieved_texts = None


    if image_url:
        image_retrieve_result = search_person_by_image(
            image_url,
            1,
            face_recognizer,
            image_vector_db
        )
        retrieved_images_paths = [image_retrieve_result[0]["match_image_path"]]
        if not retrieved_texts:
            retrieved_texts = image_retrieve_result[0]["textual_content"]
    else:
        retrieved_images_paths = None

    ans = call_generative_model(
        input_text=query,
        input_image_path=image_url,
        retrieved_texts=retrieved_texts,
        retrieved_images_paths=retrieved_images_paths,
        model_name="gemini-2.5-flash"
    )

    return ans

In [65]:
sample_text = """
هومن سیدی کیست؟
"""

RAG(sample_text)

'هومن سیدی بازیگر، کارگردان، فیلمنامه\u200cنویس و تدوین\u200cگر ایرانی است که در تاریخ ۸ آذر ۱۳۵۹ در رشت، استان گیلان متولد شد. او از دهه هشتاد تاکنون در عرصه هنر فعال بوده و در فیلم\u200cها و مجموعه\u200cهای تلویزیونی و نمایش خانگی متعددی به ایفای نقش یا کارگردانی پرداخته است.\n\n**برخی از آثار سینمایی او:**\n*   «یک تکه نان» (۱۳۸۳)\n*   «پابرهنه در بهشت» (۱۳۸۴)\n*   «چهارشنبه سوری» (۱۳۸۴)\n*   «آنکه دریا می\u200cرود» (۱۳۸۵)\n*   «زخم شانه حوا» (۱۳۸۶)\n*   «یک وجب از آسمان» (۱۳۸۷)\n*   «آفریقا» (۱۳۸۹)\n*   «آزادراه» (۱۳۸۹)\n*   «سیزده» (۱۳۹۱)\n*   «خط ویژه» (۱۳۹۲)\n*   «شب بیرون» (۱۳۹۲)\n*   «'

In [101]:
sample_text = """
تصویر داده شده مربوط به چه فردی است؟
"""

sample_image_path = str(Config.PROJECT_PATH / "homan_seyedi.jpg")

RAG(sample_text, sample_image_path)

No person name found!


'تصویر داده شده مربوط به **هومن سیدی** است.'

In [104]:
sample_text = """
شغل فرد موجود در تصویر چیست؟ آیا او تاجر است؟
"""

sample_image_path = str(Config.PROJECT_PATH / "homan_seyedi.jpg")

RAG(sample_text, sample_image_path)

No person name found!


'فرد موجود در تصویر **هومن سیدی** است.\n\nبر اساس متن بازیابی\u200cشده، شغل او **بازیگری، کارگردانی، فیلم\u200cنامه\u200cنویسی و تدوین\u200cگری** است.\n\n**خیر، او تاجر نیست.** او در عرصه هنر فعالیت دارد.'

In [105]:
sample_text = """
آیا فرد موجود در تصویر مرد است؟ آیا ازدواج کرده است؟ چند فرزند دارد؟ در کجا متولد شده است؟
"""

sample_image_path = str(Config.PROJECT_PATH / "homan_seyedi.jpg")

RAG(sample_text, sample_image_path)

No person name found!


'بله، فرد موجود در تصویر مرد است.\n\nبر اساس متن بازیابی\u200cشده، هومن سیدی در تاریخ ۸ آذر ۱۳۵۹ در رشت، استان گیلان متولد شده است.\n\nمتن بازیابی\u200cشده اطلاعاتی در مورد وضعیت تأهل یا تعداد فرزندان او ارائه نمی\u200cدهد.'